# SolProbe Google Colab T4 Demo

This notebook trains a tiny PyTorch model on Colab and streams GPU plus training metrics to a running SolProbe backend. Free Colab GPU availability is best-effort; if Colab gives you a CPU runtime, the notebook still runs but GPU utilization will be zero.

In [ ]:
# Runtime > Change runtime type > T4 GPU, then fill these in.
BACKEND_URL = "https://YOUR_PUBLIC_SOLPROBE_BACKEND"  # example: https://abc123.ngrok-free.app
API_KEY = "solprobe-demo-key"
JOB_ID = "colab-t4-demo"
NODE_ID = "colab-t4-0"


In [ ]:
import json, subprocess, time, urllib.request, urllib.error

class SolProbeColabClient:
    def __init__(self, backend_url, api_key, node_id="colab-t4-0", job_id="colab-demo", gpu_model=None):
        self.backend_url = backend_url.rstrip("/")
        self.api_key = api_key
        self.node_id = node_id
        self.job_id = job_id
        self.gpu_model = gpu_model or self.detect_gpu_model()

    def detect_gpu_model(self):
        try:
            out = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader,nounits"], text=True, timeout=1).strip()
            return out.splitlines()[0].strip() if out else "unknown"
        except Exception:
            return "unknown"

    def sample_gpu(self):
        values = {"gpu_temp_c": 0.0, "gpu_utilization_pct": 0.0, "fb_used_mb": 0.0, "fb_free_mb": 0.0, "power_usage_w": 0.0}
        try:
            out = subprocess.check_output([
                "nvidia-smi",
                "--query-gpu=temperature.gpu,utilization.gpu,memory.used,memory.free,power.draw",
                "--format=csv,noheader,nounits",
            ], text=True, timeout=1).strip()
            parts = [p.strip() for p in out.splitlines()[0].split(",")]
            values.update({
                "gpu_temp_c": float(parts[0]),
                "gpu_utilization_pct": float(parts[1]),
                "fb_used_mb": float(parts[2]),
                "fb_free_mb": float(parts[3]),
                "power_usage_w": float(parts[4]),
            })
        except Exception:
            pass
        return values

    def request(self, path, payload, method="POST"):
        req = urllib.request.Request(
            self.backend_url + path,
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json", "X-SolProbe-API-Key": self.api_key},
            method=method,
        )
        with urllib.request.urlopen(req, timeout=5) as resp:
            raw = resp.read()
        return json.loads(raw.decode("utf-8")) if raw else {}

    def register_job(self):
        return self.request("/api/v1/jobs", {
            "job_id": self.job_id,
            "name": "Google Colab T4 tiny training",
            "config": {"platform": "google-colab", "gpu_model": self.gpu_model, "model": "tiny-mlp"},
            "node_ids": [self.node_id],
        })

    def update_job_status(self, status):
        return self.request(f"/api/v1/jobs/{self.job_id}/status", {"status": status}, method="PATCH")

    def report_step(self, step, loss, grad_norm, lr, throughput_tps, mfu_pct):
        now = int(time.time() * 1000)
        gpu = self.sample_gpu()
        gpu.update({"node_id": self.node_id, "gpu_index": 0, "gpu_model": self.gpu_model, "timestamp_ms": now})
        return self.request("/api/v1/metrics/batches", {
            "gpu": [gpu],
            "training": {
                "node_id": self.node_id,
                "job_id": self.job_id,
                "timestamp_ms": now,
                "step": int(step),
                "loss": float(loss),
                "gradient_norm": float(grad_norm),
                "learning_rate": float(lr),
                "throughput_tps": float(throughput_tps),
                "mfu_pct": float(mfu_pct),
            },
        })


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
client = SolProbeColabClient(BACKEND_URL, API_KEY, node_id=NODE_ID, job_id=JOB_ID)
print("device:", device, "gpu:", client.gpu_model)
client.register_job()
client.update_job_status("running")

model = torch.nn.Sequential(
    torch.nn.Linear(64, 128), torch.nn.GELU(), torch.nn.Linear(128, 1)
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = torch.nn.BCEWithLogitsLoss()

batch_size = 512
tokens_per_batch = batch_size * 64
peak_tps = 12000.0

for step in range(80):
    start = time.perf_counter()
    x = torch.randn(batch_size, 64, device=device)
    y = (x.sum(dim=1, keepdim=True) > 0).float()
    optimizer.zero_grad(set_to_none=True)
    loss = loss_fn(model(x), y)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0).item()
    optimizer.step()
    batch_time = time.perf_counter() - start
    throughput = tokens_per_batch / max(batch_time, 1e-9)
    mfu_pct = throughput / peak_tps * 100.0
    client.report_step(step, loss.item(), grad_norm, optimizer.param_groups[0]["lr"], throughput, mfu_pct)
    if step % 10 == 0:
        print(f"step={step:03d} loss={loss.item():.4f} grad={grad_norm:.3f} tps={throughput:.0f}")
    time.sleep(0.5)

client.update_job_status("completed")
print("Done. Open the SolProbe Training page and look for", JOB_ID)
